# Colombia Open — Hurto por modalidades

Este notebook muestra cómo cargar el dataset **`delincuencia_hurto_modalidades`** (víctimas de hurto a residencias y entidades comerciales por municipio y día, 2010 a la fecha, Policía Nacional) con `colombia-open`.

> **Granularidad:** una fila por fecha + municipio + modalidad + arma/medio + género + grupo etario, con el número de hechos (~397,361 filas). **Para contar hechos, suma `cantidad` — nunca cuentes filas.**

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.7.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
modalidades = hub.load_polars("delincuencia_hurto_modalidades")

print(modalidades.shape)
modalidades.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | date | Fecha del hecho. |
| `anio` | int | Año de `fecha` — para joins con datasets anuales como `demografia`. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Códigos DIVIPOLA — join key hacia `municipios`. |
| `nombre_dpto` / `nombre_mpio` | string | Nombres como los entrega la fuente (sin el sufijo "(CT)" de las capitales). |
| `tipo_hurto` | string | Modalidad tal como la reporta la fuente (solo 2 valores: residencias / entidades comerciales). |
| `modalidad` | string | Derivado de `tipo_hurto`: `residencias` o `comercio`. |
| `arma_medio` | string | Arma o medio empleado. |
| `genero` / `grupo_etario` | string, nullable | De la víctima — mayoritariamente nulo (~53% de las filas agregadas). |
| `cantidad` | int | Número de hechos para esa combinación (siempre > 0). |

Clave primaria: `(fecha, cod_mpio, tipo_hurto, arma_medio, genero, grupo_etario)`. Esquema completo: [`docs/datasets/delincuencia_hurto_modalidades.md`](../../docs/datasets/delincuencia_hurto_modalidades.md).

## Residencias vs. comercio, por año

In [ ]:
(
    modalidades.group_by(["anio", "modalidad"])
    .agg(pl.col("cantidad").sum().alias("hechos"))
    .pivot(on="modalidad", index="anio", values="hechos")
    .sort("anio")
)

## Tasa de hurto a residencias por 100 mil habitantes (join con `demografia`)

In [ ]:
poblacion = hub.load_polars("demografia").filter(pl.col("area_geografica") == "Total")

tasas = (
    modalidades.filter(pl.col("modalidad") == "residencias")
    .group_by(["cod_mpio", "anio"])
    .agg(pl.col("cantidad").sum().alias("hechos"))
    .join(poblacion.select(["cod_mpio", "nombre_mpio", "anio", "poblacion"]), on=["cod_mpio", "anio"])
    .with_columns((pl.col("hechos") / pl.col("poblacion") * 100_000).round(1).alias("tasa_100k"))
)

# Municipios de más de 300 mil habitantes con mayor tasa en 2024
tasas.filter((pl.col("anio") == 2024) & (pl.col("poblacion") > 300_000)).sort(
    "tasa_100k", descending=True
).head(10)

## Warnings de validación y procedencia

El validador reporta como **warning** (no error) cualquier `cod_mpio` que no exista en el `municipios` vigente, y cualquier `tipo_hurto` nuevo de la fuente que el pipeline todavía no clasifique en `modalidad`.

In [ ]:
for warning in hub.get_dataset("delincuencia_hurto_modalidades")["warnings"]:
    print("warning:", warning)

hub.provenance("delincuencia_hurto_modalidades")

## Limitaciones conocidas

- **Solo dos modalidades**: residencias y entidades comerciales — no incluye hurto a personas (`delincuencia_hurto_personas`, dataset separado) ni otras modalidades (vehículos, motocicletas...).
- **`genero` es mayoritariamente nulo** (~53% de las filas agregadas) — a diferencia de lesiones personales, donde el género casi siempre se reporta.
- **Agregación fuerte respecto a la fuente cruda**: 653,384 filas crudas (un hecho = un registro) colapsan a ~397,361 filas publicadas, porque muchos hechos comparten la misma combinación de fecha + municipio + modalidad + arma + género + grupo etario.
- **1 fila sin municipio excluida** — no se fabrica un código.

Documentación completa: [`docs/datasets/delincuencia_hurto_modalidades.md`](../../docs/datasets/delincuencia_hurto_modalidades.md).